# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 08 — Model Comparison

---

### Purpose
Conduct a rigorous, systematic comparison of all trained and tuned classifiers
across all feature sets and both preprocessing pipelines. This notebook is the
central evaluation hub of the Fingerprint framework.

### Objectives
1. Load all saved models (default and tuned)
2. Re-evaluate all models on a unified held-out test set
3. Compare across **accuracy, precision, recall, F1 (macro + weighted)**
4. Measure **training time, prediction time, peak memory usage**
5. Generate the master **confusion matrix gallery**
6. Produce the master **ROC curve gallery**
7. Generate a comprehensive **comparison table**
8. Produce **interactive Plotly visualisations** for all key metrics
9. Save the master comparison table

### Workflow
```
Trained Models (from NB03–NB06 + NB07)
        │
        ▼
  Unified Evaluation on Test Set
        │
        ├── Metric Aggregation Table
        ├── Confusion Matrix Gallery
        ├── ROC Curve Gallery
        └── Interactive Visualisations
                │
                ▼
        outputs/model_comparison.csv
        figures/comparison_*.png
```

### Notebook Outline
1. Imports
2. Configuration
3. Load Feature Matrices
4. Load Saved Models
5. Unified Evaluation
6. Accuracy Comparison
7. Precision Comparison
8. Recall Comparison
9. F1 Score Comparison
10. Confusion Matrix Gallery
11. ROC Curve Gallery
12. Training Time Comparison
13. Prediction Time Comparison
14. Memory Usage Comparison
15. Comparison Heatmap
16. Master Comparison Table
17. Notebook Summary

---

## 1. Imports

In [ ]:
import sys
import logging
import warnings
import time
import tracemalloc
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering.utils import load_feature_matrix, setup_logger
from src.evaluation.evaluator import ModelEvaluator, ModelComparator
from src.visualization.plots import (
    plot_confusion_matrix,
    plot_roc_curves,
    plot_model_comparison_bar,
    plot_feature_comparison_heatmap,
)
from src.utils.helpers import (
    set_global_seed, train_test_val_split, load_yaml, make_output_dirs,
    print_section_header, save_json,
)

print('✅ All libraries imported successfully.')

---

## 2. Configuration

In [ ]:
cfg = load_yaml(PROJECT_ROOT / 'configs' / 'training.yaml')

RANDOM_SEED = cfg['random_seed']
TEST_SIZE   = cfg['evaluation']['test_size']
VAL_SIZE    = cfg['evaluation']['val_size']
FEAT_CFG    = cfg['features']

set_global_seed(RANDOM_SEED)

DIR_MODELS  = PROJECT_ROOT / cfg['output']['models_dir']
DIR_FIGURES = PROJECT_ROOT / cfg['output']['figures_dir']
DIR_OUTPUTS = PROJECT_ROOT / cfg['output']['outputs_dir']
make_output_dirs(DIR_MODELS, DIR_FIGURES, DIR_OUTPUTS)

setup_logger(str(PROJECT_ROOT / cfg['logging']['log_file']), cfg['logging']['level'])
logger = logging.getLogger(__name__)

# Evaluation metrics to display in comparison tables
METRIC_COLS = [
    'accuracy', 'precision_macro', 'recall_macro',
    'f1_macro', 'f1_weighted', 'roc_auc_macro',
    'train_time_s', 'pred_time_s', 'peak_memory_mb',
]

print('Configuration loaded.')

---

## 3. Load Feature Matrices

In [ ]:
# ── Load all feature sets — Pipeline A (Fingerprint-Preserving) ────────────────
print('Loading TF-IDF features ...')
X_tfidf, y_tfidf = load_feature_matrix(
    PROJECT_ROOT / FEAT_CFG['tfidf']['fingerprint'],
    PROJECT_ROOT / FEAT_CFG['labels']['fingerprint'],
)

print('Loading Char N-Gram features ...')
X_char, y_char = load_feature_matrix(
    PROJECT_ROOT / FEAT_CFG['char']['fingerprint'],
    PROJECT_ROOT / FEAT_CFG['labels']['fingerprint'],
)

print('Loading Stylometric features ...')
X_style, y_style = load_feature_matrix(
    PROJECT_ROOT / FEAT_CFG['style']['fingerprint'],
    PROJECT_ROOT / FEAT_CFG['labels']['fingerprint'],
)
if sp.issparse(X_style):
    X_style = X_style.toarray()

print('Loading Embedding features ...')
EMB_DIR = PROJECT_ROOT / 'data' / 'features' / 'embedding'
X_emb   = np.load(str(EMB_DIR / 'emb_fingerprint.npz'))['embeddings']
y_emb   = np.load(str(EMB_DIR / 'labels_emb_fingerprint.npy'))

# ── Class names ───────────────────────────────────────────────────────────────
classes = np.load(
    str(PROJECT_ROOT / 'data' / 'features' / 'tfidf' / 'classes_tfidf_fingerprint.npy'),
    allow_pickle=True,
)

# ── Unified test set splits (same seed for all — fair comparison) ──────────────
def get_test_split(X, y):
    """Return the test split using the global seed — ensures identical test samples."""
    _, _, X_te, _, _, y_te = train_test_val_split(
        X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
    )
    return X_te, y_te

X_te_tf,    y_te_tf    = get_test_split(X_tfidf, y_tfidf)
X_te_ch,    y_te_ch    = get_test_split(X_char,  y_char)
X_te_st,    y_te_st    = get_test_split(X_style, y_style)
X_te_em,    y_te_em    = get_test_split(X_emb,   y_emb)

print(f'\nTest set sizes — TF-IDF: {X_te_tf.shape[0]}  Char: {X_te_ch.shape[0]}')
print(f'                Style:  {X_te_st.shape[0]}  Emb:  {X_te_em.shape[0]}')
print(f'Classes: {list(classes)}')

---

## 4. Load Saved Models

In [ ]:
# ── Helper: load a saved model and its test split ─────────────────────────────
def load_model(model_path: Path):
    """Load a joblib-serialised model payload or raw estimator."""
    payload = joblib.load(model_path)
    # FingerprintClassifier payload is a dict; tuned models are raw estimators
    if isinstance(payload, dict):
        return payload['model'], payload['name'], payload.get('train_time', 0.0)
    return payload, model_path.stem, 0.0

# ── Define all models to compare ─────────────────────────────────────────────
# Format: (model_path, feature_set, X_test, y_test, label)
MODEL_REGISTRY = [
    # Logistic Regression
    (DIR_MODELS / 'logistic_regression' / 'logistic_regression_tfidf.joblib',
     'tfidf', X_te_tf, y_te_tf, 'LR / TF-IDF'),
    (DIR_MODELS / 'logistic_regression' / 'logistic_regression_char.joblib',
     'char', X_te_ch, y_te_ch, 'LR / Char'),
    (DIR_MODELS / 'logistic_regression' / 'logistic_regression_style.joblib',
     'style', X_te_st, y_te_st, 'LR / Style'),
    (DIR_MODELS / 'logistic_regression' / 'logistic_regression_embedding.joblib',
     'embedding', X_te_em, y_te_em, 'LR / Emb'),
    # Linear SVM
    (DIR_MODELS / 'linear_svm' / 'linear_svm_tfidf.joblib',
     'tfidf', X_te_tf, y_te_tf, 'SVM / TF-IDF'),
    (DIR_MODELS / 'linear_svm' / 'linear_svm_char.joblib',
     'char', X_te_ch, y_te_ch, 'SVM / Char'),
    (DIR_MODELS / 'linear_svm' / 'linear_svm_style.joblib',
     'style', X_te_st, y_te_st, 'SVM / Style'),
    (DIR_MODELS / 'linear_svm' / 'linear_svm_embedding.joblib',
     'embedding', X_te_em, y_te_em, 'SVM / Emb'),
    # Random Forest
    (DIR_MODELS / 'random_forest' / 'random_forest_tfidf.joblib',
     'tfidf', X_te_tf, y_te_tf, 'RF / TF-IDF'),
    (DIR_MODELS / 'random_forest' / 'random_forest_style.joblib',
     'style', X_te_st, y_te_st, 'RF / Style'),
    (DIR_MODELS / 'random_forest' / 'random_forest_embedding.joblib',
     'embedding', X_te_em, y_te_em, 'RF / Emb'),
    # XGBoost
    (DIR_MODELS / 'xgboost' / 'xgboost_style.joblib',
     'style', X_te_st, y_te_st, 'XGB / Style'),
    (DIR_MODELS / 'xgboost' / 'xgboost_embedding.joblib',
     'embedding', X_te_em, y_te_em, 'XGB / Emb'),
    # Tuned models
    (DIR_MODELS / 'tuned' / 'logistic_regression_tuned.joblib',
     'tfidf', X_te_tf, y_te_tf, 'LR-Tuned / TF-IDF'),
    (DIR_MODELS / 'tuned' / 'random_forest_tuned.joblib',
     'embedding', X_te_em, y_te_em, 'RF-Tuned / Emb'),
    (DIR_MODELS / 'tuned' / 'xgboost_tuned.joblib',
     'embedding', X_te_em, y_te_em, 'XGB-Tuned / Emb'),
]

# Filter to only models that exist on disk
AVAILABLE_MODELS = [(p, fs, Xte, yte, lbl) for p, fs, Xte, yte, lbl in MODEL_REGISTRY
                    if p.exists()]

print(f'Available models to compare: {len(AVAILABLE_MODELS)}')
for p, fs, _, _, lbl in AVAILABLE_MODELS:
    print(f'  ✅ {lbl:25s}  →  {p.name}')

---

## 5. Unified Evaluation

In [ ]:
# ── Evaluate all available models ─────────────────────────────────────────────
comparator = ModelComparator()

for model_path, feature_set, X_te, y_te, label in AVAILABLE_MODELS:
    estimator, model_name, train_time = load_model(model_path)

    ev = ModelEvaluator(
        model_name=label,
        feature_set=feature_set,
        classes=classes,
    )
    ev.evaluate(
        estimator=estimator,
        X_test=X_te,
        y_test=y_te,
        train_time=train_time,
    )
    comparator.add(ev)

print(f'\n✅ Evaluated {len(comparator.evaluators)} model configurations.')

In [ ]:
# ── Master comparison table ────────────────────────────────────────────────────
comparison_df = comparator.comparison_table()
comparison_df[['rank', 'model_name', 'feature_set'] + METRIC_COLS].style.highlight_max(
    subset=['accuracy', 'f1_macro', 'f1_weighted'],
    color='lightgreen',
).highlight_min(
    subset=['train_time_s', 'pred_time_s', 'peak_memory_mb'],
    color='lightyellow',
).format(precision=4)

---

## 6. Accuracy Comparison

In [ ]:
# ── Bar chart: Accuracy ────────────────────────────────────────────────────────
acc_path = DIR_FIGURES / 'comparison_accuracy.png'
plot_model_comparison_bar(
    comparison_df=comparison_df,
    metric='accuracy',
    title='Model Comparison — Accuracy (Test Set)',
    out_path=acc_path,
)

# Interactive version
fig = px.bar(
    comparison_df.sort_values('accuracy', ascending=False),
    x='model_name',
    y='accuracy',
    color='feature_set',
    title='Model Comparison — Test Set Accuracy',
    template='plotly_dark',
)
fig.update_yaxes(range=[0, 1.05])
fig.show()

---

## 7. Precision Comparison

In [ ]:
fig = px.bar(
    comparison_df.sort_values('precision_macro', ascending=False),
    x='model_name',
    y=['precision_macro', 'precision_weighted'],
    barmode='group',
    title='Model Comparison — Precision (Macro vs Weighted)',
    template='plotly_dark',
)
fig.update_yaxes(range=[0, 1.05])
fig.show()

plot_model_comparison_bar(
    comparison_df=comparison_df,
    metric='precision_macro',
    title='Model Comparison — Macro Precision (Test Set)',
    out_path=DIR_FIGURES / 'comparison_precision.png',
)

---

## 8. Recall Comparison

In [ ]:
fig = px.bar(
    comparison_df.sort_values('recall_macro', ascending=False),
    x='model_name',
    y=['recall_macro', 'recall_weighted'],
    barmode='group',
    title='Model Comparison — Recall (Macro vs Weighted)',
    template='plotly_dark',
)
fig.update_yaxes(range=[0, 1.05])
fig.show()

plot_model_comparison_bar(
    comparison_df=comparison_df,
    metric='recall_macro',
    title='Model Comparison — Macro Recall (Test Set)',
    out_path=DIR_FIGURES / 'comparison_recall.png',
)

---

## 9. F1 Score Comparison

In [ ]:
# ── Macro F1 bar chart ────────────────────────────────────────────────────────
plot_model_comparison_bar(
    comparison_df=comparison_df,
    metric='f1_macro',
    title='Model Comparison — Macro F1 Score (Test Set)',
    out_path=DIR_FIGURES / 'comparison_f1_macro.png',
)

# ── Macro vs Weighted F1 interactive chart ────────────────────────────────────
fig = px.scatter(
    comparison_df,
    x='f1_macro',
    y='f1_weighted',
    color='feature_set',
    symbol='model_name',
    text='model_name',
    size_max=12,
    title='Macro F1 vs Weighted F1 — All Models',
    template='plotly_dark',
)
fig.update_traces(textposition='top center')
fig.add_shape(type='line', x0=0, x1=1, y0=0, y1=1,
              line=dict(color='grey', dash='dash'))
fig.show()

---

## 10. Confusion Matrix Gallery

In [ ]:
# ── Generate confusion matrices for the top-5 models by f1_macro ──────────────
top5 = comparator.evaluators[:5]   # Already sorted by f1_macro

for ev in top5:
    cm = np.array(ev.results_['confusion_matrix'])
    out_path = DIR_FIGURES / f'cmp_cm_{ev.model_name.replace(" ", "_").replace("/","_")}.png'
    plot_confusion_matrix(
        cm=cm,
        class_names=list(classes),
        title=f'{ev.model_name} — Confusion Matrix',
        out_path=out_path,
        normalize=True,
    )
    print(f'  ✅ CM saved: {out_path.name}')

print(f'\nConfusion matrices generated for top {len(top5)} models.')

---

## 11. ROC Curve Gallery

In [ ]:
# ── ROC curves for models with probability output ─────────────────────────────
for ev in top5:
    if ev.results_.get('y_proba') is not None:
        out_path = DIR_FIGURES / f'cmp_roc_{ev.model_name.replace(" ", "_").replace("/","_")}.png'
        plot_roc_curves(
            y_test=ev.results_['y_test'],
            y_proba=ev.results_['y_proba'],
            class_names=list(classes),
            title=f'{ev.model_name} — ROC Curves',
            out_path=out_path,
        )
        print(f'  ✅ ROC saved: {out_path.name}')

---

## 12. Training Time Comparison

In [ ]:
fig = px.bar(
    comparison_df.sort_values('train_time_s'),
    x='train_time_s',
    y='model_name',
    orientation='h',
    color='feature_set',
    title='Model Comparison — Training Time (seconds, lower is better)',
    template='plotly_dark',
    labels={'train_time_s': 'Training Time (s)', 'model_name': 'Model'},
)
fig.show()

plot_model_comparison_bar(
    comparison_df=comparison_df,
    metric='train_time_s',
    title='Training Time Comparison',
    out_path=DIR_FIGURES / 'comparison_train_time.png',
)

---

## 13. Prediction Time Comparison

In [ ]:
fig = px.bar(
    comparison_df.sort_values('pred_time_s'),
    x='pred_time_s',
    y='model_name',
    orientation='h',
    color='feature_set',
    title='Model Comparison — Prediction Time (seconds, lower is better)',
    template='plotly_dark',
    labels={'pred_time_s': 'Prediction Time (s)', 'model_name': 'Model'},
)
fig.show()

---

## 14. Memory Usage Comparison

In [ ]:
fig = px.scatter(
    comparison_df,
    x='f1_macro',
    y='peak_memory_mb',
    color='feature_set',
    symbol='model_name',
    size='f1_macro',
    size_max=20,
    title='F1 Macro vs Peak Memory — Efficiency Frontier',
    template='plotly_dark',
    labels={'f1_macro': 'Macro F1', 'peak_memory_mb': 'Peak Memory (MB)'},
)
fig.show()

---

## 15. Comparison Heatmap

In [ ]:
# ── Multi-metric heatmap ───────────────────────────────────────────────────────
heatmap_metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted']

plot_feature_comparison_heatmap(
    comparison_df=comparison_df,
    metrics=heatmap_metrics,
    title='Model × Metric Performance Heatmap (All Models)',
    out_path=DIR_FIGURES / 'comparison_heatmap.png',
)

# ── Interactive heatmap with Plotly ───────────────────────────────────────────
pivot = comparison_df.set_index('model_name')[heatmap_metrics]

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=heatmap_metrics,
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=0, zmax=1,
    text=np.round(pivot.values, 4),
    texttemplate='%{text}',
))
fig.update_layout(
    title='Model × Metric Heatmap',
    template='plotly_dark',
    height=max(400, len(pivot) * 40),
)
fig.show()

---

## 16. Master Comparison Table

In [ ]:
# ── Save master CSV ────────────────────────────────────────────────────────────
comparator.save_comparison_table(DIR_OUTPUTS / 'model_comparison.csv')
print('✅ Master comparison table saved → outputs/model_comparison.csv')

# ── Full display ───────────────────────────────────────────────────────────────
display_df = comparison_df[['rank', 'model_name', 'feature_set'] + METRIC_COLS].copy()
display_df = display_df.rename(columns={
    'model_name':       'Model',
    'feature_set':      'Features',
    'accuracy':         'Acc',
    'precision_macro':  'Prec↑',
    'recall_macro':     'Rec↑',
    'f1_macro':         'F1↑',
    'f1_weighted':      'WF1↑',
    'roc_auc_macro':    'AUC↑',
    'train_time_s':     'Train(s)↓',
    'pred_time_s':      'Pred(s)↓',
    'peak_memory_mb':   'Mem(MB)↓',
})
display_df.style.highlight_max(
    subset=['Acc', 'Prec↑', 'Rec↑', 'F1↑', 'WF1↑', 'AUC↑'],
    color='lightgreen',
).highlight_min(
    subset=['Train(s)↓', 'Pred(s)↓', 'Mem(MB)↓'],
    color='lightyellow',
).format(precision=4)

---

## 17. Notebook Summary

### Comparison Findings

| Dimension | Winner | Notes |
|---|---|---|
| **Accuracy** | *(run to populate)* | *(run to populate)* |
| **Macro F1** | *(run to populate)* | *(run to populate)* |
| **Weighted F1** | *(run to populate)* | *(run to populate)* |
| **ROC-AUC** | *(run to populate)* | *(run to populate)* |
| **Fastest Training** | *(run to populate)* | *(run to populate)* |
| **Lowest Memory** | *(run to populate)* | *(run to populate)* |
| **Best Feature Set** | *(run to populate)* | *(run to populate)* |

### Saved Artefacts
| File | Description |
|---|---|
| `outputs/model_comparison.csv` | Master comparison table |
| `figures/comparison_*.png` | Metric bar charts |
| `figures/comparison_heatmap.png` | Multi-metric heatmap |
| `figures/cmp_cm_*.png` | Top-5 confusion matrices |
| `figures/cmp_roc_*.png` | Top-5 ROC curves |

→ **Notebook 09**: Best Model Selection — rank, justify, analyse

---
*Fingerprint Project — Model Comparison — Complete*